# 04 弦截法与 Müller 抛物线法

Newton 法的主要成本是导数。若 $f'(x)$ 难以解析给出，或者导数计算比函数值昂贵，可以用若干个函数值构造局部近似：

* 弦截法（secant method）用两点割线近似导数；
* Müller 法用三点抛物线近似函数，并取该抛物线的零点作为下一次迭代。

两者都属于开方法，不保持括区间。它们通常比二分法快，但也继承了初值敏感和分母退化等风险。


In [ ]:
import math
import pathlib
import sys

import numpy as np

ROOT = pathlib.Path.cwd()
while ROOT.name != "py-sc" and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import muller_method, newton_method, secant_method


## 弦截法

Newton 法使用

$$
x_{k+1}=x_k-\frac{f(x_k)}{f'(x_k)}.
$$

弦截法用两点差商替代导数：

$$
f'(x_k)\approx \frac{f(x_k)-f(x_{k-1})}{x_k-x_{k-1}},
$$

从而得到

$$
x_{k+1}=x_k-f(x_k)\frac{x_k-x_{k-1}}{f(x_k)-f(x_{k-1})}.
$$

它每步只需要一个新函数值，简单根附近的理论收敛阶约为黄金比例 $1.618$，介于线性收敛和 Newton 二次收敛之间。


In [ ]:
def teaching_secant(f, x0, x1, tolerance=1e-12, max_iterations=30):
    history = [float(x0), float(x1)]
    f0 = float(f(x0))
    f1 = float(f(x1))
    for k in range(1, max_iterations + 1):
        denominator = f1 - f0
        if abs(denominator) <= 1e-14:
            raise ValueError("secant denominator is too small")
        x2 = history[-1] - f1 * (history[-1] - history[-2]) / denominator
        f2 = float(f(x2))
        history.append(x2)
        if abs(f2) <= tolerance:
            return x2, k, True, np.array(history)
        f0, f1 = f1, f2
    return history[-1], max_iterations, False, np.array(history)

f = lambda x: math.cos(x) - x
root, iterations, converged, history = teaching_secant(f, 0.0, 1.0)
official = secant_method(f, 0.0, 1.0, tolerance=1e-12)
newton = newton_method(f, lambda x: -math.sin(x) - 1.0, 0.5, tolerance=1e-12)

print(f"teaching secant root = {root:.15f}, iterations = {iterations}, converged = {converged}")
print(f"official secant root = {official.root:.15f}, residual = {official.residual:.3e}")
print(f"Newton comparison iterations = {newton.iterations}")
print("secant history:", np.array2string(history, precision=12))


## Müller 抛物线法

弦截法用直线穿过两个点，Müller 法则用三个点构造插值抛物线。设当前三点为 $x_0,x_1,x_2$，令抛物线以 $x_2$ 为局部原点表示为

$$
p(t)=a t^2 + b t + c,\qquad t=x-x_2.
$$

取离 $x_2$ 更近且数值上稳定的零点，可写成

$$
x_3=x_2-\frac{2c}{b+\operatorname{sign}(b)\sqrt{b^2-4ac}}.
$$

完整 Müller 法可进入复平面。本章的公共实现保持实数标量结果，因此当判别式为负时显式报错。


In [ ]:
def cubic(x):
    return x**3 - x - 2.0

muller = muller_method(cubic, 0.0, 1.0, 2.0, tolerance=1e-12)
secant_cubic = secant_method(cubic, 1.0, 2.0, tolerance=1e-12)

print(f"Muller root = {muller.root:.15f}, iterations = {muller.iterations}, residual = {muller.residual:.3e}")
print(f"secant cubic root = {secant_cubic.root:.15f}, iterations = {secant_cubic.iterations}")
print("Muller history:", np.array2string(muller.history, precision=12))


## 退化与失败情形

无导数方法虽然减少了导数需求，但并不自动稳健：

* 弦截法若 $f(x_k)=f(x_{k-1})$，分母为零；
* Müller 法若三个节点重复，差商无意义；
* 实数 Müller 法遇到负判别式时，抛物线的零点进入复平面；
* 若初值跨越多个根或函数局部形状复杂，迭代可能收敛到非预期根或发散。

因此，实际流程常先用区间扫描定位候选根，再选用弦截、Newton 或 Müller 等开方法加速。


In [ ]:
try:
    secant_method(lambda x: x**2 + 1.0, -1.0, 1.0)
except ValueError as exc:
    print("secant failure:", exc)

try:
    muller_method(lambda x: x**2 + 1.0, -1.0, 0.0, 1.0)
except ValueError as exc:
    print("Muller failure:", exc)


## 本节小结

弦截法用差商替代导数，是 Newton 法的自然无导数版本，简单根附近通常超线性收敛。Müller 法进一步用三点抛物线捕捉函数曲率，在某些问题上可比弦截法更快，并能自然扩展到复根计算。这里的实现保持实数根教学范围，因此对负判别式显式失败。无导数开方法适合作为括区间扫描后的加速器，而不是全局可靠的根隔离工具。
